# Partie 2 – Flux streaming : mobilité urbaine (vélos en libre-service)

**Source** : CityBikes API (`api.citybik.es/v2`), réseaux français des 8 villes de `config.py`.
**Collecte** : `p2_citybikes_stream.py`, un relevé de toutes les stations toutes les 120 s → `data/raw/citybikes/date=YYYY-MM-DD/snapshots.csv`.

**Limite** : CityBikes ne fournit que l'état en temps réel des stations. L'historique « depuis l'existence du réseau » n'est pas accessible via cette API ; l'étude porte sur la période de collecte (~20 h).

> À relancer en entier (*Kernel → Restart & Run All*) à la fin de la collecte.

In [ ]:
import sys, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from config import CITIES
from figures.style import apply_style, save_fig, CITY_COLORS, STATE_COLORS, PALETTE

try:
    from urbanhub_utils import check_city_hourly
except ImportError:  # repli si urbanhub_utils.py n'est pas encore sur main
    def check_city_hourly(df):
        assert {"city", "hour"} <= set(df.columns), "colonnes city/hour manquantes"
        bad = set(df["city"].dropna()) - set(CITIES)
        assert not bad, f"villes hors config.py : {bad}"
        assert str(df["hour"].dt.tz) == "UTC", "hour doit être en UTC"
        assert (df["hour"] == df["hour"].dt.floor("h")).all(), "hour non arrondie à l'heure"
        assert not df.duplicated(["city", "hour"]).any(), "doublons (city, hour)"
        print(f"check_city_hourly OK : {len(df)} lignes, {df['city'].nunique()} villes")
        return df

apply_style()
SRC = "Source : CityBikes API – collecte UrbanHub"
RAW = ROOT / "data" / "raw" / "citybikes"
CLEAN = ROOT / "data" / "clean"
CLEAN.mkdir(parents=True, exist_ok=True)

## 1. Chargement et nettoyage

- Concaténation de tous les relevés ; `poll_time` (heure du relevé) en UTC.
- Dédoublonnage sur (`station_id`, `poll_time`).
- `capacity = bikes_available + free_slots` ; stations de capacité 0 (fermées/hors service) retirées.
- `occupancy = bikes_available / capacity` (0 = vide, 1 = pleine).
- `delta = |Δ bikes_available|` entre deux relevés consécutifs d'une station = estimation des vélos pris ou rendus.
- `empty` / `full` : station sans vélo / sans place libre.

In [ ]:
files = sorted(glob.glob(str(RAW / "*" / "snapshots.csv")))
raw = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
print(f"{len(files)} fichier(s), {len(raw):,} lignes brutes")

b = raw.copy()
b["poll_time"] = pd.to_datetime(b["poll_time"], utc=True, format="ISO8601")
b = b.dropna(subset=["station_id", "poll_time", "bikes_available", "free_slots"])
b = b.drop_duplicates(["station_id", "poll_time"])
b["capacity"] = b["bikes_available"] + b["free_slots"]
b = b[b["capacity"] > 0]
b = b[b["city"].isin(CITIES)]
b["occupancy"] = b["bikes_available"] / b["capacity"]
b = b.sort_values(["station_id", "poll_time"])
b["delta"] = b.groupby("station_id")["bikes_available"].diff().abs()
b["empty"] = b["bikes_available"] == 0
b["full"] = b["free_slots"] == 0
b["hour"] = b["poll_time"].dt.floor("h")

n_polls = b["poll_time"].nunique()
duree = (b["poll_time"].max() - b["poll_time"].min()).total_seconds() / 3600
print(f"{len(b):,} lignes propres ({len(raw) - len(b):,} retirées)")
print(f"{b['station_id'].nunique():,} stations, {b['city'].nunique()} villes, {n_polls} relevés sur {duree:.1f} h")
print(f"Période : {b['poll_time'].min():%d/%m %H:%M} → {b['poll_time'].max():%d/%m %H:%M} UTC")
b.groupby("city")["station_id"].nunique().sort_values(ascending=False).rename("stations")

In [ ]:
b.to_parquet(CLEAN / "bikes.parquet", index=False)

# Table ville × heure pour la partie 4 (heures incomplètes de début/fin retirées)
first_h, last_h = b["hour"].min(), b["hour"].max()
hourly = (b[(b["hour"] > first_h) & (b["hour"] < last_h)]
          .groupby(["city", "hour"])
          .agg(movements=("delta", "sum"), occupancy=("occupancy", "mean"))
          .reset_index())
hourly = check_city_hourly(hourly)
hourly.to_parquet(CLEAN / "bikes_city_hourly.parquet", index=False)
hourly.head()

## 2. Indicateurs par station
`usage_rate = mouvements / capacité` : nombre de « rotations » par place sur la période (comparable entre petites et grandes stations).

In [ ]:
st = (b.groupby(["station_id", "station_name", "city"])
       .agg(lat=("latitude", "first"), lon=("longitude", "first"),
            cap=("capacity", "median"), moves=("delta", "sum"),
            occ=("occupancy", "mean"), empty=("empty", "mean"), full=("full", "mean"))
       .reset_index())
st["usage_rate"] = st["moves"] / st["cap"]
st.describe()[["cap", "moves", "usage_rate", "occ", "empty", "full"]].round(2)

## Q1 – Quelles stations présentent le plus fort taux d'utilisation ?
**Méthode** : classement des stations par `usage_rate` (mouvements / capacité).

In [ ]:
top = st.nlargest(15, "usage_rate").iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.barh(top["station_name"].str[:40] + " (" + top["city"] + ")", top["usage_rate"],
        color=[CITY_COLORS[c] for c in top["city"]])
ax.set_title("Q1 – Top 15 des stations au plus fort taux d'utilisation")
ax.set_xlabel(f"Taux d'utilisation (mouvements / place, sur {duree:.0f} h)")
ax.grid(axis="y", visible=False)
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=CITY_COLORS[c], label=c) for c in top["city"].unique()],
          title="Ville", loc="lower right")
save_fig("p2_q1_top_stations", source=SRC)
plt.show()

best = st.loc[st["usage_rate"].idxmax()]
print(f"Chiffre clé : « {best.station_name} » ({best.city}) = {best.usage_rate:.1f} rotations/place "
      f"vs médiane {st['usage_rate'].median():.2f}")
st.groupby("city")["usage_rate"].median().sort_values(ascending=False).round(2).rename("usage médian")

**Interprétation** : *[à compléter : quelles stations / quels quartiers dominent (gares, universités, centres d'affaires ?), écart avec la médiane]*

## Q2 – Peut-on identifier des zones où l'offre de vélos est insuffisante ?
**Méthode** : part du temps où chaque station est vide ; stations vides > 20 % du temps ; regroupement par zone (grille de ~1 km : lat/lon arrondies à 0,01°).

In [ ]:
st["zone"] = st["lat"].round(2).astype(str) + ", " + st["lon"].round(2).astype(str)
zones = (st.groupby(["city", "zone"])
           .agg(stations=("station_id", "count"), pct_vide=("empty", "mean"),
                st_vides_20=("empty", lambda s: (s > 0.2).sum()))
           .reset_index())
zones = zones[zones["stations"] >= 2].sort_values("pct_vide", ascending=False)

z = zones.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(10, 6.5))
ax.barh(z["city"] + " – " + z["zone"], z["pct_vide"] * 100, color=[CITY_COLORS[c] for c in z["city"]])
ax.axvline(20, color=PALETTE["accent"], ls="--", lw=1.5, label="Seuil 20 %")
ax.set_title("Q2 – Zones (~1 km) où les stations sont le plus souvent vides")
ax.set_xlabel("Part du temps sans vélo disponible (%)")
ax.grid(axis="y", visible=False)
ax.legend(loc="lower right")
save_fig("p2_q2_zones_vides", source=SRC)
plt.show()

n20 = (st["empty"] > 0.2).sum()
print(f"Chiffre clé : {n20} stations ({n20 / len(st):.0%}) vides plus de 20 % du temps ; "
      f"{(zones['pct_vide'] > 0.2).sum()} zones en offre insuffisante")
zones.head(10)

**Interprétation** : *[à compléter : localisation des zones (périphérie ? collines ?), lien probable avec le relief ou les flux domicile–travail]*

## Q3 – Quels sont les pics d'utilisation journaliers ?
**Méthode** : mouvements par heure (table ville × heure) divisés par le nombre de stations de la ville, puis moyenne par heure de la journée en heure de Paris. Les heures non couvertes par la collecte restent vides.

In [ ]:
h = hourly.copy()
h["h_local"] = h["hour"].dt.tz_convert("Europe/Paris").dt.hour
n_st = st.groupby("city")["station_id"].count()
h["mv_station"] = h["movements"] / h["city"].map(n_st)          # comparable entre villes
prof = h.groupby(["h_local", "city"])["mv_station"].mean().unstack().reindex(range(24))

fig, ax = plt.subplots()
for c in prof.columns:
    ax.plot(prof.index, prof[c], marker="o", ms=4, color=CITY_COLORS[c], label=c)
for a, z_ in [(8, 9), (17, 19)]:
    ax.axvspan(a - 0.5, z_ + 0.5, color=PALETTE["light"], zorder=0)
ax.set_title("Q3 – Profil journalier d'utilisation des vélos")
ax.set_xlabel("Heure de la journée (heure de Paris)")
ax.set_ylabel("Mouvements par station et par heure")
ax.set_xticks(range(0, 24, 2))
ax.legend(title="Ville", ncol=2)
save_fig("p2_q3_profil_horaire", source=SRC)
plt.show()

tot = h.groupby("h_local")["movements"].mean()
print(f"Chiffre clé : pic à {tot.idxmax()} h ({tot.max():,.0f} mouvements/h toutes villes), "
      f"creux à {tot.idxmin()} h ({tot.min():,.0f})")

**Interprétation** : *[à compléter : pics du matin et du soir = trajets domicile–travail ? Attention : avec ~20 h de collecte, chaque heure n'est observée qu'une fois]*

## Q4 – Existe-t-il des déséquilibres géographiques dans la disponibilité des vélos ?
**Méthode** : occupation moyenne de chaque station de Paris sur une carte (rouge < 20 % = presque vide, orange, vert > 80 % = presque pleine).

In [ ]:
import folium
paris = st[st["city"] == "Paris"]
def couleur(o):
    return STATE_COLORS["low"] if o < 0.2 else STATE_COLORS["high"] if o > 0.8 else STATE_COLORS["mid"]

m = folium.Map(CITIES["Paris"], zoom_start=12, tiles="cartodbpositron")
for r in paris.itertuples():
    folium.CircleMarker([r.lat, r.lon], radius=4, weight=0, fill=True, fill_opacity=0.8,
                        fill_color=couleur(r.occ),
                        tooltip=f"{r.station_name} – occupation {r.occ:.0%}").add_to(m)
m.get_root().html.add_child(folium.Element(
    "<div style='position:fixed;bottom:20px;left:20px;z-index:999;background:white;padding:8px 12px;"
    "border-radius:6px;font:13px sans-serif'><b>Occupation moyenne</b><br>"
    f"<span style='color:{STATE_COLORS['low']}'>●</span> &lt; 20 % (vide)<br>"
    f"<span style='color:{STATE_COLORS['mid']}'>●</span> 20–80 %<br>"
    f"<span style='color:{STATE_COLORS['high']}'>●</span> &gt; 80 % (pleine)</div>"))
m.save(str(ROOT / "figures" / "p2_q4_carte_paris.html"))

# version PNG pour le rapport
fig, ax = plt.subplots(figsize=(8, 7))
sc = ax.scatter(paris["lon"], paris["lat"], c=paris["occ"] * 100, cmap="RdYlGn", vmin=0, vmax=100, s=12)
fig.colorbar(sc, ax=ax, label="Occupation moyenne (%)")
ax.set_title("Q4 – Disponibilité des vélos par station à Paris")
ax.set_xlabel("Longitude (°)"); ax.set_ylabel("Latitude (°)")
ax.set_aspect(1 / np.cos(np.radians(48.86)))
save_fig("p2_q4_carte_paris", source=SRC)
plt.show()

c_ = paris[["lat", "lon"]].mean()
paris = paris.assign(secteur=np.where(paris["lat"] > c_["lat"], "Nord", "Sud") + "-"
                     + np.where(paris["lon"] > c_["lon"], "Est", "Ouest"))
print("Chiffre clé – occupation moyenne par secteur :")
print((paris.groupby("secteur")["occ"].mean() * 100).round(1).astype(str) + " %")
m

**Interprétation** : *[à compléter : centre plein / périphérie vide (ou l'inverse) ? Rive droite / rive gauche ? Effet du relief (Montmartre, Belleville)]*

## Q5 – Peut-on détecter des stations critiques nécessitant un rééquilibrage ?
**Méthode** : station vide > 30 % du temps → **à recharger** ; pleine > 30 % du temps → **à vider**.

In [ ]:
st["action"] = np.select([st["empty"] > 0.3, st["full"] > 0.3], ["À recharger", "À vider"], "OK")
crit = st[st["action"] != "OK"].sort_values(["action", "empty", "full"], ascending=[True, False, False])

cnt = st.groupby(["city", "action"]).size().unstack(fill_value=0)
cnt = cnt.reindex(columns=["À recharger", "À vider"], fill_value=0)
ax = cnt.plot.bar(color=[STATE_COLORS["low"], STATE_COLORS["high"]], figsize=(10, 5))
ax.set_title("Q5 – Stations critiques à rééquilibrer par ville")
ax.set_xlabel(""); ax.set_ylabel("Nombre de stations")
ax.legend(title="Action")
plt.xticks(rotation=0)
save_fig("p2_q5_stations_critiques", source=SRC)
plt.show()

print(f"Chiffre clé : {(st.action == 'À recharger').sum()} stations à recharger, "
      f"{(st.action == 'À vider').sum()} à vider ({len(crit) / len(st):.0%} du parc)")
crit[["city", "station_name", "cap", "empty", "full", "action"]].head(20)

**Interprétation** : *[à compléter : où envoyer les camions de rééquilibrage en priorité ; lien avec Q2 et Q4]*

## Conclusion partie 2
*[3–4 phrases : stations et zones clés, rythme journalier, recommandations de rééquilibrage, limite de la durée de collecte]*